In [ ]:
! pip freeze

In [ ]:
# === BLS音声: 2本比較ダッシュボード（診断→処方） ==========================
# 依存: pip install openai-whisper jiwer pyloudnorm webrtcvad librosa soundfile noisereduce pydub fugashi ipadic
import os, numpy as np, pandas as pd, librosa, soundfile as sf, warnings, json, math
warnings.filterwarnings("ignore")
import pyloudnorm as pyln
import webrtcvad
import noisereduce as nr
from jiwer import cer, wer
from pydub import AudioSegment
try:
    import whisper, torch
    HAS_WHISPER = True
except Exception:
    HAS_WHISPER = False

# ====== 設定 ======
BAD_PATH  = "/root/ikadai/0604data/2回目_右前.wav"   # ダメだった
GOOD_PATH = "/root/ikadai/0604data/5回目_右前.wav"   # 比較的良い
REF_BAD   = ""   # ← 正解文（分かる範囲で貼る。未入力ならCERはNaN）
REF_GOOD  = ""   # ← 正解文

BLS_PROMPT = "傷病者, 周囲の安全, 感染防御"

# ====== ユーティリティ ======
def load_mono16k(path, sr=16000):
    y, sr = librosa.load(path, sr=sr, mono=True)
    return y, sr

def lufs(y, sr):
    meter = pyln.Meter(sr)
    return meter.integrated_loudness(y)

def vad_speech_ratio(y, sr, frame_ms=30, aggressiveness=2):
    vad = webrtcvad.Vad(aggressiveness)
    frame_len = int(sr * frame_ms / 1000)
    n = max(1, len(y) // frame_len)
    speech = 0
    for i in range(n):
        frm = y[i*frame_len:(i+1)*frame_len]
        pcm16 = (np.clip(frm, -1, 1) * 32767).astype(np.int16).tobytes()
        speech += vad.is_speech(pcm16, sr)
    return speech / n

def band_ratio(y, sr, lo, hi, ref_lo=300, ref_hi=4000):
    S = np.abs(librosa.stft(y, n_fft=1024, hop_length=256))**2
    freqs = librosa.fft_frequencies(sr=sr, n_fft=1024)
    def pow_band(a,b):
        idx = (freqs>=a) & (freqs<b)
        return float(S[idx,:].sum()) + 1e-9
    ref = pow_band(ref_lo, ref_hi)
    return pow_band(lo, hi) / ref

def hum_peaks(y, sr, mains=(50,60), harmonics=3, tol=1.0):
    # 50/60Hzと高調波が目立つか簡易検知
    import scipy.signal as sig
    f, Pxx = sig.welch(y, sr, nperseg=4096)
    flags = {}
    for f0 in mains:
        peaks = []
        for h in range(1, harmonics+1):
            target = f0*h
            if target >= sr/2: break
            idx = (f>=target-tol) & (f<=target+tol)
            band = Pxx[idx].mean() if idx.any() else 0.0
            ref  = Pxx[(f>=300)&(f<=4000)].mean() + 1e-12
            peaks.append(band/ref)
        flags[f0] = max(peaks) if peaks else 0.0
    return flags  # 比が>5〜10で要ノッチ目安

def snr_coarse(y, sr, vad_ratio):
    # 粗SNR: 全体RMSと(1-vad_ratio)でノイズRMS近似
    rms_all = np.sqrt(np.mean(y**2)+1e-12)
    noise_rms = rms_all * (1 - vad_ratio + 1e-3)
    return 20*np.log10((rms_all+1e-9)/(noise_rms+1e-9))

# ---- 前処理（処方） ----
def normalize_lufs(y, sr, target=-20.0):
    meter = pyln.Meter(sr)
    loud = meter.integrated_loudness(y)
    gain = 10**((target - loud)/20)
    return np.clip(y*gain, -1, 1)

def highpass(y, sr, fc=90, order=4):
    from scipy.signal import butter, filtfilt
    b,a = butter(order, fc/(sr/2), btype='high')
    return filtfilt(b,a,y)

def notch_hum(y, sr, freqs=(50,100,150,200,60,120,180), Q=30):
    from scipy.signal import iirnotch, filtfilt
    out = y.copy()
    for f0 in freqs:
        if f0 < sr/2:
            b,a = iirnotch(w0=f0/(sr/2), Q=Q)
            out = filtfilt(b,a,out)
    return out

def denoise(y, sr, noise_sec=6, prop=0.6):
    n = min(len(y), int(noise_sec*sr))
    y_noise = y[:n] if n>0 else y[: int(2*sr)]
    return nr.reduce_noise(y=y, y_noise=y_noise, sr=sr, prop_decrease=prop)

def clean_chain(y, sr, lvl=True, hp=True, hum=True, nr_on=True):
    if lvl: y = normalize_lufs(y, sr, -20.0)
    if hum: y = notch_hum(y, sr)
    if hp:  y = highpass(y, sr, 90)
    if nr_on: y = denoise(y, sr, prop=0.6)
    return np.clip(y, -1, 1)

# ---- Whisper ----
def transcribe(path, prompt=""):
    if not HAS_WHISPER:
        return "", {"avg_logprob":np.nan, "compression_ratio":np.nan, "no_speech_prob":np.nan, "duration":np.nan}
    model = whisper.load_model("large-v3-turbo", device="cuda" if torch.cuda.is_available() else "cpu")
    res = model.transcribe(
        path,
        language="ja",
        temperature=0.0,
        beam_size=10,
        best_of=5,
        condition_on_previous_text=False,
        word_timestamps=True,
        initial_prompt=prompt
    )

    segs = res.get("segments", [])
    meta = {
        "avg_logprob": np.nanmean([s.get("avg_logprob", np.nan) for s in segs]) if segs else np.nan,
        "compression_ratio": np.nanmean([s.get("compression_ratio", np.nan) for s in segs]) if segs else np.nan,
        "no_speech_prob": np.nanmean([s.get("no_speech_prob", np.nan) for s in segs]) if segs else np.nan,
        "duration": res.get("duration", np.nan)
    }
    return res.get("text",""), meta

def cer_safe(ref, hyp):
    if not ref or not hyp: return np.nan
    return cer("".join(ref.split()), "".join(hyp.split()))

# ---- 1ファイル診断 ----
def diagnose_one(path, ref_text="", apply_clean=False, save_clean=False):
    y, sr = load_mono16k(path)
    raw_lufs = lufs(y, sr)
    vad_ratio = vad_speech_ratio(y, sr)
    snr = snr_coarse(y, sr, vad_ratio)
    high_ratio = band_ratio(y, sr, 4000, min(sr/2, 8000))
    low_bump  = band_ratio(y, sr, 0, 120)
    hum = hum_peaks(y, sr)

    wav_for_asr = path
    if apply_clean:
        y2 = clean_chain(y, sr, lvl=True, hp=True, hum=True, nr_on=True)
        wav_for_asr = os.path.splitext(path)[0] + "_clean_tmp.wav"
        sf.write(wav_for_asr, y2, sr)
        if save_clean:
            sf.write(os.path.splitext(path)[0] + "_clean_saved.wav", y2, sr)

    hyp, meta = transcribe(wav_for_asr, prompt=BLS_PROMPT)
    cer_val = cer_safe(ref_text, hyp)
    cps = len(hyp.replace(" ","")) / max(meta.get("duration") or (len(y)/sr), 1e-6)

    return {
        "path": path,
        "LUFS": raw_lufs,
        "VAD_speech_ratio": vad_ratio,
        "SNR_est_dB": snr,
        "HighBandRatio_>4k": high_ratio,
        "LowBump_<120Hz": low_bump,
        "Hum50_ratio": hum.get(50, 0.0),
        "Hum60_ratio": hum.get(60, 0.0),
        "avg_logprob": meta["avg_logprob"],
        "compression_ratio": meta["compression_ratio"],
        "no_speech_prob": meta["no_speech_prob"],
        "chars_per_sec": cps,
        "CER": cer_val,
        "hyp_text": hyp[:200]  # プレビュー
    }

# ====== 実行: 素の音 と クリーニング後 で2本を比較 ======
rows = []
for p, ref in [(BAD_PATH, REF_BAD), (GOOD_PATH, REF_GOOD)]:
    rows.append(diagnose_one(p, ref, apply_clean=False))
for p, ref in [(BAD_PATH, REF_BAD), (GOOD_PATH, REF_GOOD)]:
    rows.append(diagnose_one(p, ref, apply_clean=True, save_clean=False))

df = pd.DataFrame(rows)
df.insert(1, "cond", ["raw","raw","clean","clean"])
display(df[["path","cond","LUFS","SNR_est_dB","VAD_speech_ratio",
            "HighBandRatio_>4k","LowBump_<120Hz","Hum50_ratio","Hum60_ratio",
            "avg_logprob","compression_ratio","no_speech_prob","chars_per_sec","CER","hyp_text"]])

# ====== 自動所見（しきい値ベース） ======
def comment(row):
    tips = []
    if row["LUFS"] <= -28: tips.append("音量が小さい→ -20 LUFSへ正規化推奨")
    if row["SNR_est_dB"] < 12: tips.append("SNRが低い→ ノイズ低減 or 近接収音推奨")
    if row["LowBump_<120Hz"] > 0.25: tips.append("低域衝撃が強い→ HPF 80–120Hz推奨")
    if (row["Hum50_ratio"]>8) or (row["Hum60_ratio"]>8): tips.append("ハム目立つ→ 50/60Hzノッチ")
    if row["HighBandRatio_>4k"] < 0.08: tips.append("高域不足→ ローパス厳禁、必要なら3.5–5kHzを軽く持ち上げ")
    if np.isnan(row["CER"]) and HAS_WHISPER and row["avg_logprob"]<-0.5:
        tips.append("Whisper信頼度低→ beam拡大/分割粒度調整")
    return " / ".join(tips) if tips else "特に顕著な問題は検知されず（要耳確認）"

df["auto_comment"] = df.apply(comment, axis=1)
display(df[["path","cond","auto_comment"]])


In [ ]:
# ==== 音量底上げ → Whisper文字起こし（最小） ====
# 依存: pip install openai-whisper pyloudnorm librosa soundfile jiwer
import numpy as np, librosa, soundfile as sf
import pyloudnorm as pyln
from jiwer import cer
import whisper, torch

IN_WAV  = "/root/ikadai/0604data/2回目_右前.wav"   # 低音量の方
OUT_WAV = "/root/ikadai/0604data/2回目_右前_norm.wav"
REF     = ""  # 正解文があれば入れる（なければ空でOK）

BLS_PROMPT = "傷病者, 周囲の安全, 感染防御"#, 意識なし, 反応なし, 呼吸なし, 胸骨圧迫, 気道確保, AED, ショック, 解析中, 電極, 救急要請, 人工呼吸, 30回, 2回, 交代, 安全確認"

def normalize_lufs(y, sr, target=-20.0):
    meter = pyln.Meter(sr)
    loud = meter.integrated_loudness(y)
    gain = 10 ** ((target - loud) / 20)
    y_ = np.clip(y * gain, -1, 1)
    return y_, loud, target

# 1) 読み込み & 正規化（-20 LUFS）
y, sr = librosa.load(IN_WAV, sr=16000, mono=True)
y_norm, before_lufs, target_lufs = normalize_lufs(y, sr, -20.0)
sf.write(OUT_WAV, y_norm, sr)
print(f"Normalized: {before_lufs:.1f} LUFS → {target_lufs:.1f} LUFS  (保存先: {OUT_WAV})")

# 2) Whisperで文字起こし
model = whisper.load_model("large-v3", device="cuda" if torch.cuda.is_available() else "cpu")
res = model.transcribe(
    OUT_WAV,
    language="ja",
    temperature=0.0,
    beam_size=10,
    best_of=5,
    condition_on_previous_text=False,
    word_timestamps=True,
    initial_prompt=BLS_PROMPT,
)
text = res.get("text","")
segs = res.get("segments", [])
avg_logprob = np.mean([s.get("avg_logprob", np.nan) for s in segs]) if segs else np.nan
compression_ratio = np.mean([s.get("compression_ratio", np.nan) for s in segs]) if segs else np.nan

print("\n=== Whisper 出力（先頭300文字）===")
print(text[:300])
print("\nmeta: avg_logprob =", avg_logprob, " / compression_ratio =", compression_ratio)

# 3) 参考: CER（正解がある場合だけ）
if REF:
    ref_c = "".join(REF.split()); hyp_c = "".join(text.split())
    print("CER =", cer(ref_c, hyp_c))
else:
    print("CER: 正解文REFが未設定のためスキップ")

Normalized: -26.4 LUFS → -20.0 LUFS  (保存先: /root/ikadai/0604data/2回目_右前_norm.wav)

=== Whisper 出力（先頭300文字）===
新型コロナウイルス感染症については、 新型コロナウイルス感染症については、

meta: avg_logprob = -0.4815647125244141  / compression_ratio = 1.5352112676056338
CER: 正解文REFが未設定のためスキップ


In [4]:
# === 必要なら事前インストール ===
# pip install SpeechRecognition pydub librosa noisereduce soundfile

import os
import numpy as np
import librosa
import soundfile as sf
import noisereduce as nr
from pydub import AudioSegment, effects
import speech_recognition as sr

# ===== 設定 =====
WAV_IN = "/root/ikadai/0604data/2回目_右前.wav"
WORK_SR = 16000              # 処理サンプリング周波数（Whisper等と互換性◎）
OUT_DIR = os.path.dirname(WAV_IN)

DO_NORMALIZE_LIBROSA = True  # librosaでRMSノーマライズ
DO_NORMALIZE_PYDUB   = False # pydubでピークノーマライズ（上とどちらかでOK）
TARGET_RMS_DBFS = -20.0      # RMSターゲット（例：-20 dBFS）

DO_NOISE_REDUCE = True       # noisereduceを使う
NOISE_SAMPLE_SEC = 3.0       # 冒頭のn秒をノイズ見本として使用（調整可）

USE_HIGHPASS = False         # 低域ハム/足音対策のハイパス（任意）
HP_CUTOFF_HZ = 80.0

USE_LOWPASS = False          # 高周波ザラつき対策のローパス（任意）
LP_CUTOFF_HZ = 6000.0

# ===== ユーティリティ =====
def rms_dbfs(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    rms = np.sqrt(np.mean(np.square(x))) + 1e-12
    return 20 * np.log10(rms)

def apply_rms_normalize(x: np.ndarray, target_dbfs: float) -> np.ndarray:
    cur = rms_dbfs(x)
    gain_db = target_dbfs - cur
    gain = 10 ** (gain_db / 20)
    y = x * gain
    # クリップ回避のソフトクリップ
    y = np.clip(y, -0.999, 0.999)
    return y

def apply_pydub_normalize(path_in: str, path_out: str):
    seg = AudioSegment.from_file(path_in)
    seg = effects.normalize(seg)  # ピーク基準
    seg.set_frame_rate(WORK_SR)
    seg.export(path_out, format="wav")

def butter_filter(x, sr, cutoff, btype="high"):
    from scipy.signal import butter, filtfilt
    nyq = sr * 0.5
    b, a = butter(4, cutoff/nyq, btype=btype)
    return filtfilt(b, a, x).astype(np.float32)

# ===== 前処理パイプライン =====
def preprocess(wav_in: str, work_sr: int) -> str:
    y, sr = librosa.load(wav_in, sr=work_sr, mono=True)  # float32, -1.0〜1.0

    # オプション：フィルタ
    if USE_HIGHPASS:
        y = butter_filter(y, work_sr, HP_CUTOFF_HZ, "high")
    if USE_LOWPASS:
        y = butter_filter(y, work_sr, LP_CUTOFF_HZ, "low")

    # ノイズ低減
    if DO_NOISE_REDUCE:
        n = int(NOISE_SAMPLE_SEC * work_sr)
        y_noise = y[:max(n, 1)]
        y = nr.reduce_noise(y=y, y_noise=y_noise, sr=work_sr, prop_decrease=0.8)

    # 音量ノーマライズ（どちらか1つでOK）
    if DO_NORMALIZE_LIBROSA:
        y = apply_rms_normalize(y, TARGET_RMS_DBFS)
        tmp_path = os.path.join(OUT_DIR, "tmp_norm_librosa.wav")
        sf.write(tmp_path, y, work_sr, subtype="PCM_16")
    elif DO_NORMALIZE_PYDUB:
        tmp_raw = os.path.join(OUT_DIR, "tmp_preNR.wav")
        sf.write(tmp_raw, y, work_sr, subtype="PCM_16")
        tmp_path = os.path.join(OUT_DIR, "tmp_norm_pydub.wav")
        apply_pydub_normalize(tmp_raw, tmp_path)
        os.remove(tmp_raw)
    else:
        tmp_path = os.path.join(OUT_DIR, "tmp_noproc.wav")
        sf.write(tmp_path, y, work_sr, subtype="PCM_16")

    return tmp_path

# ===== SpeechRecognition でASR =====
def transcribe_google(wav_path: str, lang="ja-JP") -> str:
    r = sr.Recognizer()
    with sr.AudioFile(wav_path) as source:
        audio = r.record(source)
    try:
        return r.recognize_google(audio, language=lang)
    except sr.UnknownValueError:
        return "(認識できませんでした)"
    except sr.RequestError as e:
        return f"(APIリクエストに失敗: {e})"

if __name__ == "__main__":
    out_path = preprocess(WAV_IN, WORK_SR)
    print(f"[INFO] 前処理済みファイル: {out_path}")

    text = transcribe_google(out_path, lang="ja-JP")
    print("=== 認識結果 ===")
    print(text)



/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] 前処理済みファイル: /root/ikadai/0604data/tmp_norm_librosa.wav
=== 認識結果 ===
(認識できませんでした)
